## Trạng thái bản lưu trên GitHub

Đã chạy full 3 seed, 64 outer run ngày 30/09/2026. Xem results/notebook_07; không thay model web 06.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

# 07 — Kiểm chứng motion và attention theo đề cương
Notebook độc lập, không cần chạy 04/04b/05/06 trước. Giữ kiến trúc một luồng 4 block, 33 khớp, 32 mốc; sửa pipeline v2 có version. Không thay model web 06.

**Đăng ký trước cho lượt này:** A0/A1/A2/A3 cùng recipe và weighted CE cân bằng lớp; A3_cost_225 là thí nghiệm chi phí riêng. Ba seed đã chốt, 4 outer × 3 inner subject. Epoch theo validation loss; threshold chỉ từ inner validation. Báo cáo fixed, balanced, sensitivity trên cùng xác suất. Không chọn seed tốt nhất. Output smoke không phải kết quả khoa học.

Các outer-test cũ đã được xem để phát triển phương pháp: đừng gọi toàn bộ quá trình này là test hoàn toàn chưa từng sử dụng. Cần tập ngoài mới cho xác nhận cuối cùng.

**Input:** gmdcsa24_pose_32.npz, gmdcsa24_features_32.npz, gmdcsa24_pose_manifest.csv và gmdcsa24_cross_subject_folds.csv. Có thể gắn output notebook 01 hoặc upload gói 07-inputs-verified.zip đi kèm. ZIP .npz đổi đuôi và CSV .txt cũ cũng được hỗ trợ. Không cần video gốc hoặc MediaPipe để train lại.

## 1. Môi trường
Bật GPU Kaggle. Không cài lại PyTorch nếu Kaggle đã có. Full gồm 240 lượt fit mạng tối đa 50 epoch/lượt, thường dừng sớm; không hứa thời gian hoàn tất. SVM có thêm tìm kiếm inner riêng.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import importlib.util
required = ["numpy", "pandas", "torch", "sklearn"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Missing packages: {missing}; use a Kaggle Python GPU notebook")
import torch
torch.set_num_threads(2)
print("PyTorch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Đọc và kiểm tra dữ liệu
Giữ nguyên 160 mẫu, không loại clip khó. Tái tạo eight-channel feature từ pose; dùng detection mask để visibility bằng 0 ở mẫu thiếu. XYZ đã được nội suy từ notebook 01, không thể khôi phục đầy đủ đoạn mất pose gốc từ file 32 frame.

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


CHANNELS = ["relative_x", "relative_y", "visibility", "joint_velocity_x", "joint_velocity_y",
            "torso_angle", "hip_displacement_x", "hip_displacement_y"]
PAIRS = [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16),
         (17, 18), (19, 20), (21, 22), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)]
FEATURE_VERSION = "v2-mask-visibility-raw-pose-augmentation"


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def find_unique(root, names):
    matches = sorted({path for name in names for path in Path(root).rglob(name)})
    if not matches:
        raise FileNotFoundError(f"Missing input: {names}; attach notebook 01 outputs")
    hashes = {sha256(path) for path in matches}
    if len(hashes) != 1:
        raise ValueError(f"Conflicting copies; remove ambiguous inputs: {matches}")
    return matches[0]


def load_inputs(root):
    paths = {
        "pose": find_unique(root, ["gmdcsa24_pose_32.npz", "gmdcsa24_pose_32.zip"]),
        "features": find_unique(root, ["gmdcsa24_features_32.npz", "gmdcsa24_features_32.zip"]),
        "manifest": find_unique(root, ["gmdcsa24_pose_manifest.csv", "pose_mnf.txt"]),
        "folds": find_unique(root, ["gmdcsa24_cross_subject_folds.csv", "cross.txt"]),
    }
    with np.load(paths["pose"], allow_pickle=False) as archive:
        data = {key: archive[key].copy() for key in archive.files}
    with np.load(paths["features"], allow_pickle=False) as archive:
        features = {key: archive[key].copy() for key in archive.files}
    for key in ["video_ids", "subjects", "labels", "masks"]:
        if not np.array_equal(data[key], features[key]):
            raise ValueError(f"Pose/feature alignment failed: {key}")
    manifest = pd.read_csv(paths["manifest"])
    folds = pd.read_csv(paths["folds"])
    for column, key in [("video_id", "video_ids"), ("subject", "subjects"), ("label_id", "labels")]:
        if not np.array_equal(manifest[column].to_numpy(), data[key]):
            raise ValueError(f"Manifest order mismatch: {column}")
    count = len(data["labels"])
    if count != 160 or data["poses"].shape != (count, 32, 33, 4):
        raise ValueError("This preregistered experiment expects 160 poses of shape (32,33,4)")
    if len(np.unique(data["video_ids"])) != count or set(np.unique(data["subjects"])) != {1, 2, 3, 4}:
        raise ValueError("Duplicate IDs or unexpected subjects")
    if set(np.unique(data["labels"])) != {0, 1} or data["masks"].shape != (count, 32):
        raise ValueError("Invalid labels or masks")
    if not np.isfinite(data["poses"]).all() or not np.isfinite(features["durations"]).all() or np.any(features["durations"] <= 0):
        raise ValueError("Nonfinite pose or invalid duration")
    if len(folds) != count * 4 or folds["fold"].nunique() != 4:
        raise ValueError("Saved split file does not have four folds")
    for _, group in folds.groupby("fold"):
        indices = group["sample_index"].to_numpy(dtype=int)
        if not np.array_equal(np.sort(indices), np.arange(count)):
            raise ValueError("Invalid saved sample indices")
        if not np.array_equal(group["video_id"].to_numpy(), data["video_ids"][indices]):
            raise ValueError("Saved fold IDs do not match pose order")
        if not np.array_equal(group["subject"].to_numpy(), data["subjects"][indices]) or not np.array_equal(group["label_id"].to_numpy(), data["labels"][indices]):
            raise ValueError("Saved fold subjects or labels mismatch")
        partitions = [set(group.loc[group["split"] == split, "subject"]) for split in ["train", "val", "test"]]
        if any(partitions[left] & partitions[right] for left, right in [(0, 1), (0, 2), (1, 2)]):
            raise ValueError("Subject overlap in saved folds")
    data.update(durations=features["durations"].astype(np.float32), manifest=manifest,
                provenance={key: {"path": str(path), "sha256": sha256(path)} for key, path in paths.items()})
    return data


def make_features_v2(pose, detected, duration, mask_visibility=True):
    pose = np.asarray(pose, dtype=np.float32)
    detected = np.asarray(detected, dtype=bool)
    if pose.shape != (32, 33, 4) or detected.shape != (32,) or not np.isfinite(pose).all():
        raise ValueError("Invalid finite pose/mask shape")
    if not np.isfinite(duration) or duration <= 0:
        raise ValueError("Invalid duration")
    hips = pose[:, [23, 24], :2].mean(axis=1)
    shoulders = pose[:, [11, 12], :2].mean(axis=1)
    length = np.linalg.norm(shoulders - hips, axis=-1)
    usable = length[length > 1e-3]
    scale = max(float(np.median(usable)) if len(usable) else 1.0, 1e-3)
    relative = (pose[..., :2] - hips[:, None, :]) / scale
    velocity = np.zeros_like(relative)
    velocity[1:] = np.diff(relative, axis=0) / max(float(duration) / 31, 1e-6)
    visibility = np.clip(pose[..., 3], 0, 1).copy()
    if mask_visibility:
        visibility[~detected] = 0
    torso = shoulders - hips
    angle = np.arctan2(torso[:, 0], -torso[:, 1]) / np.pi
    displacement = hips - hips[:1]
    local = np.concatenate([relative, visibility[..., None], velocity], axis=-1)
    global_values = np.concatenate([angle[:, None], displacement], axis=-1)
    output = np.concatenate([local, np.repeat(global_values[:, None], 33, axis=1)], axis=-1)
    return output.transpose(2, 0, 1).astype(np.float32)


def augment_pose(pose, detected, duration, rng, crop=True, flip=True, noise=True, dropout=True):
    pose, detected = pose.copy(), detected.copy()
    if crop and rng.random() < 0.5:
        length = int(rng.integers(26, 33))
        start = int(rng.integers(0, 33 - length))
        positions = np.linspace(start, start + length - 1, 32)
        lower, upper = np.floor(positions).astype(int), np.ceil(positions).astype(int)
        fraction = (positions - lower).astype(np.float32)[:, None, None]
        pose = pose[lower] * (1 - fraction) + pose[upper] * fraction
        detected = detected[lower] & detected[upper]
        duration *= (length - 1) / 31
    if flip and rng.random() < 0.5:
        pose[..., 0] = 1 - pose[..., 0]
        original = pose.copy()
        for left, right in PAIRS:
            pose[:, left], pose[:, right] = original[:, right], original[:, left]
    if noise and rng.random() < 0.35:
        torso_length = np.linalg.norm(pose[:, [11, 12], :2].mean(axis=1) - pose[:, [23, 24], :2].mean(axis=1), axis=-1)
        amplitude = max(float(np.median(torso_length)), 1e-3) * 0.01
        pose[..., :2] += rng.normal(0, amplitude, pose[..., :2].shape).astype(np.float32)
    dropped = np.empty(0, dtype=int)
    if dropout and rng.random() < 0.2:
        dropped = rng.choice(33, int(rng.integers(1, 4)), replace=False)
    return pose, detected, float(duration), dropped


def svm_features(pose, detected, duration):
    sequence = make_features_v2(pose, detected, duration)
    speed = np.linalg.norm(sequence[3:5], axis=0)
    height = np.ptp(sequence[1], axis=1)
    height_velocity = np.diff(height, prepend=height[0]) / (float(duration) / 31)
    per_frame = np.stack([speed.mean(axis=1), np.quantile(speed, 0.9, axis=1), sequence[5, :, 0],
                          sequence[6, :, 0], sequence[7, :, 0], height, height_velocity], axis=1)
    return np.concatenate([per_frame.mean(axis=0), per_frame.std(axis=0), per_frame.min(axis=0),
                           per_frame.max(axis=0), *np.quantile(per_frame, [0.1, 0.5, 0.9], axis=0)]).astype(np.float32)

In [ ]:
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd() / "Kaggle_kiem_tra_2026-09-25"
data = load_inputs(INPUT_ROOT)
print("Samples", len(data["labels"]), "Fall", int(data["labels"].sum()), "Subjects", np.unique(data["subjects"]))
print("Missing sampled frames", int((~data["masks"]).sum()))
display(data["manifest"].head())

## 3. Graph, augmentation và mô hình
Crop pose rồi tính lại duration và toàn bộ motion; noise/flip trước tính feature. Joint dropout chỉ xóa các kênh cục bộ của joint, không làm mất các kênh toàn thân phát lặp. Input v2 không tương thích mặc định với checkpoint 06: không tự thay model web.

In [ ]:
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset



EDGES = [(0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
         (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
         (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
         (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),
         (27,29),(28,30),(29,31),(30,32),(27,31),(28,32)]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_adjacency():
    adjacency = np.eye(33, dtype=np.float32)
    for source, target in EDGES:
        adjacency[source, target] = adjacency[target, source] = 1
    inverse = np.maximum(adjacency.sum(axis=1), 1e-8) ** -0.5
    return torch.from_numpy(inverse[:, None] * adjacency * inverse[None])


class SpatialGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency):
        super().__init__()
        self.projection = nn.Conv2d(in_channels, out_channels, 1)
        self.register_buffer("adjacency", adjacency.clone())
        self.edge_importance = nn.Parameter(torch.ones_like(adjacency))

    def forward(self, values):
        return torch.einsum("nctv,vw->nctw", self.projection(values), self.adjacency * self.edge_importance)


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency, stride=1, dropout=0.3):
        super().__init__()
        self.graph_conv = SpatialGraphConv(in_channels, out_channels, adjacency)
        self.temporal_conv = nn.Sequential(nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (9, 1), stride=(stride, 1), padding=(4, 0)),
            nn.BatchNorm2d(out_channels), nn.Dropout(dropout))
        self.residual = nn.Identity() if in_channels == out_channels and stride == 1 else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=(stride, 1)), nn.BatchNorm2d(out_channels))
        self.activation = nn.ReLU(inplace=True)

    def forward(self, values):
        return self.activation(self.temporal_conv(self.graph_conv(values)) + self.residual(values))


class SkeletonModel(nn.Module):
    def __init__(self, channels=8, attention=True, dropout=0.3):
        super().__init__()
        adjacency = make_adjacency()
        self.data_bn = nn.BatchNorm1d(channels * 33)
        self.blocks = nn.Sequential(STGCNBlock(channels, 64, adjacency, dropout=dropout),
            STGCNBlock(64, 64, adjacency, dropout=dropout),
            STGCNBlock(64, 128, adjacency, stride=2, dropout=dropout),
            STGCNBlock(128, 128, adjacency, dropout=dropout))
        self.temporal_attention = nn.Sequential(nn.Conv1d(128, 64, 1), nn.Tanh(), nn.Conv1d(64, 1, 1)) if attention else None
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, 2))

    def forward(self, values):
        batch, channels, frames, nodes = values.shape
        values = values.permute(0, 3, 1, 2).contiguous().view(batch, nodes * channels, frames)
        values = self.data_bn(values).view(batch, nodes, channels, frames).permute(0, 2, 3, 1)
        temporal = self.blocks(values).mean(dim=3)
        weights = torch.softmax(self.temporal_attention(temporal), dim=2) if self.temporal_attention is not None else torch.ones_like(temporal[:, :1]) / temporal.shape[2]
        return self.classifier((temporal * weights).sum(dim=2)), weights.squeeze(1)


class PoseDataset(Dataset):
    def __init__(self, data, indices, channels, augment, seed):
        self.data, self.indices, self.channels, self.augment = data, indices, channels, augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, offset):
        index = self.indices[offset]
        pose, mask, duration = self.data["poses"][index], self.data["masks"][index], self.data["durations"][index]
        dropped = []
        if self.augment:
            pose, mask, duration, dropped = augment_pose(pose, mask, duration, self.rng)
        features = make_features_v2(pose, mask, duration)
        features[:5, :, dropped] = 0
        return torch.from_numpy(features[:self.channels]), torch.tensor(int(self.data["labels"][index]), dtype=torch.long)


def make_loader(data, indices, variant, config, augment, seed):
    return DataLoader(PoseDataset(data, indices, variant["channels"], augment, seed),
                      batch_size=config["batch_size"], shuffle=augment, num_workers=0,
                      generator=torch.Generator().manual_seed(seed))

## 4. Huấn luyện và đánh giá thống nhất
Mỗi outer subject giữ riêng. Inner chọn epoch bằng unweighted CE. Refit ba subject rồi đánh giá outer một lần cho các chính sách đã chốt. SVM dùng scaler fit trên train, 49 thống kê motion và RBF; đây là baseline v2, không ghi đè số baseline cũ.

In [ ]:
import json
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch import nn



VARIANTS = {
    "A0_pose_average": {"channels": 3, "attention": False, "fall_multiplier": 1.0},
    "A1_motion_average": {"channels": 8, "attention": False, "fall_multiplier": 1.0},
    "A2_pose_attention": {"channels": 3, "attention": True, "fall_multiplier": 1.0},
    "A3_motion_attention": {"channels": 8, "attention": True, "fall_multiplier": 1.0},
    "A3_cost_225": {"channels": 8, "attention": True, "fall_multiplier": 2.25},
}


def default_config(mode="full"):
    if mode not in {"full", "smoke"}:
        raise ValueError("mode must be full or smoke")
    return {"mode": mode, "feature_version": FEATURE_VERSION, "seeds": [42, 142, 242] if mode == "full" else [42],
            "variants": list(VARIANTS), "max_epochs": 50 if mode == "full" else 2,
            "patience": 10 if mode == "full" else 1, "batch_size": 16, "learning_rate": 0.001,
            "weight_decay": 0.0001, "dropout": 0.3, "scheduler": None,
            "target_sensitivity": 0.95, "minimum_specificity": 0.75,
            "threshold_grid": np.linspace(0.05, 0.95, 37).tolist(), "allow_cpu_full": False,
            "epoch_selection": "minimum unweighted inner-validation cross entropy",
            "protocol": "4 outer subjects; 3 inner subjects; refit outer train; no final all-subject retrain"}


def metrics(labels, predictions):
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    ratio = lambda top, bottom: float(top / bottom) if bottom else None
    sensitivity = ratio(true_positive, true_positive + false_negative)
    specificity = ratio(true_negative, true_negative + false_positive)
    return {"accuracy": ratio(true_positive + true_negative, len(labels)),
            "balanced_accuracy": (sensitivity + specificity) / 2 if sensitivity is not None and specificity is not None else None,
            "sensitivity": sensitivity, "specificity": specificity,
            "precision": ratio(true_positive, true_positive + false_positive),
            "f1": ratio(2 * true_positive, 2 * true_positive + false_positive + false_negative),
            "f2": ratio(5 * true_positive, 5 * true_positive + false_positive + 4 * false_negative),
            "false_alarm_rate": ratio(false_positive, false_positive + true_negative),
            "tp": int(true_positive), "tn": int(true_negative), "fp": int(false_positive), "fn": int(false_negative)}


def select_thresholds(labels, scores, config, grid=None, fixed=0.5):
    candidates = []
    for threshold in config["threshold_grid"] if grid is None else grid:
        values = metrics(labels, scores >= threshold)
        meets_recall = values["sensitivity"] >= config["target_sensitivity"]
        feasible = meets_recall and values["specificity"] >= config["minimum_specificity"]
        tier = 2 if feasible else 1 if meets_recall else 0
        safety_rank = (tier, values["specificity"], values["f2"], -abs(threshold - fixed)) if tier else (0, values["sensitivity"], values["specificity"], -abs(threshold - fixed))
        candidates.append({"threshold": float(threshold), "metrics": values, "feasible": feasible,
                           "balanced_rank": (values["balanced_accuracy"], -abs(threshold - fixed)), "safety_rank": safety_rank})
    selected = {"fixed": {"threshold": fixed, "metrics": metrics(labels, scores >= fixed), "feasible": None},
                "balanced": max(candidates, key=lambda item: item["balanced_rank"]),
                "sensitivity": max(candidates, key=lambda item: item["safety_rank"])}
    return selected


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(".pending")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def predict(model, loader, device):
    model.eval()
    labels, probabilities, attention = [], [], []
    with torch.no_grad():
        for features, target in loader:
            logits, weights = model(features.to(device))
            labels.extend(target.tolist())
            probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().tolist())
            attention.extend(weights.cpu().tolist())
    return np.asarray(labels), np.asarray(probabilities), np.asarray(attention)


def fit_model(data, train_indices, validation_indices, variant, config, seed, device, fixed_epochs=None):
    set_seed(seed)
    model = SkeletonModel(variant["channels"], variant["attention"], config["dropout"]).to(device)
    train_loader = make_loader(data, train_indices, variant, config, True, seed)
    counts = np.bincount(data["labels"][train_indices], minlength=2)
    if np.any(counts == 0):
        raise ValueError("Training fold has a missing class")
    class_weights = len(train_indices) / (2 * counts.astype(np.float32))
    class_weights[1] *= variant["fall_multiplier"]
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    validation_loader = make_loader(data, validation_indices, variant, config, False, seed) if validation_indices is not None else None
    best_loss, best_state, best_epoch, stale = float("inf"), None, 0, 0
    history = []
    for epoch in range(1, (fixed_epochs or config["max_epochs"]) + 1):
        model.train()
        total_loss = 0
        for features, target in train_loader:
            features, target = features.to(device), target.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(features)
            loss = criterion(logits, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(target)
        record = {"epoch": epoch, "train_weighted_loss": total_loss / len(train_indices)}
        if validation_loader is not None:
            labels, scores, _ = predict(model, validation_loader, device)
            scores = np.clip(scores, 1e-7, 1 - 1e-7)
            validation_loss = float(-(labels * np.log(scores) + (1 - labels) * np.log(1 - scores)).mean())
            record["validation_unweighted_loss"] = validation_loss
            if validation_loss < best_loss - 1e-6:
                best_loss, best_epoch, stale = validation_loss, epoch, 0
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            else:
                stale += 1
        history.append(record)
        if validation_loader is not None and stale >= config["patience"]:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch if validation_loader is not None else fixed_epochs, history


def run_neural(data, config, output, device):
    subjects = sorted(np.unique(data["subjects"]).tolist())
    for variant_name in config["variants"]:
        variant = VARIANTS[variant_name]
        for seed in config["seeds"]:
            for outer in subjects:
                name = f"{variant_name}_seed{seed}_subject{outer}"
                target = output / f"{name}.json"
                if target.exists():
                    print("Resume completed:", name, flush=True)
                    continue
                train_indices = np.flatnonzero(data["subjects"] != outer)
                test_indices = np.flatnonzero(data["subjects"] == outer)
                inner_scores, inner_labels, inner_epochs, histories, splits = [], [], [], [], []
                for inner in subjects:
                    if inner == outer:
                        continue
                    fit_indices = np.flatnonzero((data["subjects"] != outer) & (data["subjects"] != inner))
                    val_indices = np.flatnonzero(data["subjects"] == inner)
                    assert not set(fit_indices) & set(val_indices) and not set(train_indices) & set(test_indices)
                    run_seed = seed + outer * 1000 + inner * 10
                    model, best_epoch, history = fit_model(data, fit_indices, val_indices, variant, config, run_seed, device)
                    labels, scores, _ = predict(model, make_loader(data, val_indices, variant, config, False, run_seed), device)
                    inner_scores.extend(scores.tolist())
                    inner_labels.extend(labels.tolist())
                    inner_epochs.append(best_epoch)
                    histories.extend([{**row, "inner_subject": inner} for row in history])
                    splits.append({"inner_subject": inner, "train_indices": fit_indices.tolist(), "val_indices": val_indices.tolist(), "best_epoch": best_epoch})
                    del model
                choices = select_thresholds(np.asarray(inner_labels), np.asarray(inner_scores), config)
                epochs = max(1, int(np.median(inner_epochs)))
                model, _, refit_history = fit_model(data, train_indices, None, variant, config, seed + outer * 1000 + 999, device, epochs)
                labels, scores, attention = predict(model, make_loader(data, test_indices, variant, config, False, seed), device)
                predictions = []
                fold_metrics = []
                for policy, choice in choices.items():
                    threshold = choice["threshold"]
                    predicted = (scores >= threshold).astype(int)
                    fold_metrics.append({"variant": variant_name, "seed": seed, "subject": outer, "policy": policy,
                                         "threshold": threshold, "epochs": epochs, "inner_target_met": choice["feasible"], **metrics(labels, predicted)})
                    for index, label, score, prediction in zip(test_indices, labels, scores, predicted):
                        predictions.append({"sample_index": int(index), "video_id": str(data["video_ids"][index]), "subject": outer,
                            "label": int(label), "score": float(score), "prediction": int(prediction), "threshold": threshold,
                            "variant": variant_name, "seed": seed, "policy": policy})
                checkpoint = {"model_state_dict": {key: value.cpu() for key, value in model.state_dict().items()},
                              "variant": variant, "feature_version": FEATURE_VERSION, "train_indices": train_indices.tolist(),
                              "test_indices": test_indices.tolist(), "config": config, "epochs": epochs}
                torch.save(checkpoint, output / f"{name}.pt")
                np.savez_compressed(output / f"{name}_attention.npz", indices=test_indices, attention=attention)
                pd.DataFrame(histories).to_csv(output / f"{name}_inner_history.csv", index=False)
                pd.DataFrame(refit_history).to_csv(output / f"{name}_refit_history.csv", index=False)
                write_json(target, {"complete": True, "inner_splits": splits, "inner_labels": inner_labels, "inner_scores": inner_scores,
                                    "choices": choices, "metrics": fold_metrics, "predictions": predictions,
                                    "parameters": sum(parameter.numel() for parameter in model.parameters())})
                print("Completed", name, "epochs", epochs, flush=True)
                del model


def run_svm(data, config, output):
    values = np.stack([svm_features(pose, mask, duration) for pose, mask, duration in zip(data["poses"], data["masks"], data["durations"])])
    subjects = sorted(np.unique(data["subjects"]).tolist())
    for outer in subjects:
        target = output / f"SVM_v2_seed0_subject{outer}.json"
        if target.exists():
            continue
        candidates = []
        grid = [(1, "scale")] if config["mode"] == "smoke" else [(regularization, gamma) for regularization in [0.1, 1, 10] for gamma in ["scale", 0.01]]
        for regularization, gamma in grid:
            labels_all, scores_all = [], []
            for inner in subjects:
                if inner == outer:
                    continue
                train = np.flatnonzero((data["subjects"] != outer) & (data["subjects"] != inner))
                validation = np.flatnonzero(data["subjects"] == inner)
                estimator = make_pipeline(StandardScaler(), SVC(C=regularization, gamma=gamma, class_weight="balanced"))
                estimator.fit(values[train], data["labels"][train])
                scores_all.extend(estimator.decision_function(values[validation]).tolist())
                labels_all.extend(data["labels"][validation].tolist())
            score = metrics(np.asarray(labels_all), np.asarray(scores_all) >= 0)["balanced_accuracy"]
            candidates.append({"C": regularization, "gamma": gamma, "rank": score, "labels": labels_all, "scores": scores_all})
        best = max(candidates, key=lambda item: item["rank"])
        threshold_grid = np.unique(np.r_[np.quantile(best["scores"], np.linspace(0, 1, 37)), 0])
        choices = select_thresholds(np.asarray(best["labels"]), np.asarray(best["scores"]), config, threshold_grid, fixed=0)
        train, test = np.flatnonzero(data["subjects"] != outer), np.flatnonzero(data["subjects"] == outer)
        estimator = make_pipeline(StandardScaler(), SVC(C=best["C"], gamma=best["gamma"], class_weight="balanced"))
        estimator.fit(values[train], data["labels"][train])
        scores = estimator.decision_function(values[test])
        predictions, fold_metrics = [], []
        for policy, choice in choices.items():
            predicted = scores >= choice["threshold"]
            fold_metrics.append({"variant": "SVM_v2", "seed": 0, "subject": outer, "policy": policy,
                                 "threshold": choice["threshold"], "inner_target_met": choice["feasible"], **metrics(data["labels"][test], predicted)})
            for index, score, prediction in zip(test, scores, predicted):
                predictions.append({"sample_index": int(index), "video_id": str(data["video_ids"][index]), "subject": outer,
                    "label": int(data["labels"][index]), "score": float(score), "prediction": int(prediction), "threshold": choice["threshold"],
                    "variant": "SVM_v2", "seed": 0, "policy": policy})
        write_json(target, {"complete": True, "choices": choices, "search": candidates, "metrics": fold_metrics, "predictions": predictions,
                            "note": "49 motion statistics; scaler fit on training subjects only; scores are margins, not probabilities"})
        print("Completed SVM_v2 subject", outer, flush=True)


def summarize_run(config, output):
    paths = sorted(output.glob("*_seed*_subject*.json"))
    expected = len(config["variants"]) * len(config["seeds"]) * 4 + 4
    if len(paths) != expected:
        raise ValueError(f"Only {len(paths)}/{expected} complete outer runs; no final summary")
    reports = [json.loads(path.read_text(encoding="utf-8")) for path in paths]
    predictions = pd.DataFrame([row for report in reports for row in report["predictions"]])
    folds = pd.DataFrame([row for report in reports for row in report["metrics"]])
    pooled = []
    for (variant, seed, policy), group in predictions.groupby(["variant", "seed", "policy"]):
        if len(group) != 160 or group["sample_index"].nunique() != 160:
            raise ValueError("OOF coverage failed; do not pool duplicates across seeds")
        pooled.append({"variant": variant, "seed": int(seed), "policy": policy, **metrics(group["label"], group["prediction"])})
    pooled = pd.DataFrame(pooled)
    for name, frame in [("predictions", predictions), ("fold_metrics", folds), ("pooled_by_seed", pooled),
                        ("errors", predictions[predictions["label"] != predictions["prediction"]])]:
        frame.to_csv(output / f"{name}.csv", index=False)
    columns = ["accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "f2", "false_alarm_rate"]
    pooled.groupby(["variant", "policy"])[columns].agg(["mean", "std"]).to_csv(output / "seed_mean_std.csv")
    write_json(output / "completion.json", {"complete": True, "mode": config["mode"], "scientific_result": config["mode"] == "full",
               "outer_runs": len(paths), "warning": "Smoke results are software checks only. Repeated seeds do not increase independent sample count. Prior outer results have already informed development."})
    return pooled


def run_experiment(data, config, output, code_hash):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu" and config["mode"] == "full" and not config["allow_cpu_full"]:
        raise RuntimeError("Full experiment requires Kaggle GPU; choose smoke locally, or explicitly allow CPU full")
    signature = {"config": config, "inputs": {key: value["sha256"] for key, value in data["provenance"].items()}, "code_hash": code_hash,
                 "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__}, "device": device}
    lock = output / "locked_protocol.json"
    if lock.exists() and json.loads(lock.read_text(encoding="utf-8")) != signature:
        raise ValueError("Run configuration/code/data/environment changed; use a NEW output directory")
    write_json(lock, signature)
    write_json(output / "input_provenance.json", data["provenance"])
    started = time.monotonic()
    run_neural(data, config, output, device)
    run_svm(data, config, output)
    summary = summarize_run(config, output)
    write_json(output / "timing.json", {"seconds_this_invocation": time.monotonic() - started, "device": device})
    return summary

## 5. Khóa cấu hình trước khi chạy
Full: 3 seed × 5 cấu hình × 4 outer fold, thêm SVM. A0–A3 là ablation kiến trúc; A3_cost_225 là phân tích cost. Mục tiêu sensitivity 0.95, specificity tối thiểu 0.75 chỉ là tiêu chí validation, không phải cam kết. Nếu không đạt sẽ ghi inner_target_met=False. Số tối đa 50 epoch không có nghĩa mọi model cần train đủ 50.

In [ ]:
RUN_MODE = "full"
config = default_config(RUN_MODE)
CODE_HASH = "711aef817db2407c5ca9cdc1cb28cfc60ffee4db617b9fe9fc310f3733567a3b"
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "output/research07"
RUN_DIR = OUTPUT_ROOT / ("07_controlled_ablation_" + RUN_MODE)
print(json.dumps(config, indent=2))
print("Output:", RUN_DIR)
if RUN_MODE == "full" and not torch.cuda.is_available():
    raise RuntimeError("Enable Kaggle GPU before full run. Local smoke must use a separate output.")

## 6. Chạy thí nghiệm
Sau mỗi outer fold, checkpoint, history và kết quả được lưu. Chạy lại cùng thư mục/cấu hình tiếp tục các fold chưa xong. Không sửa dữ liệu/code/config để ghi đè lượt cũ. Nếu Kaggle tạo session mới, gắn output version trước rồi chép thư mục lượt chạy về working trước khi tiếp tục; không coi input read-only là output.

In [ ]:
summary = run_experiment(data, config, RUN_DIR, CODE_HASH)
display(summary)
print("SMOKE ONLY — NOT RESEARCH RESULTS" if RUN_MODE == "smoke" else "Full run completed; inspect all folds, not only pooled metrics")

## 7. Xuất kết quả
Giữ pooled_by_seed.csv, seed_mean_std.csv, fold_metrics.csv, predictions.csv, errors.csv và histories. Không gộp ba seed thành 480 người/video độc lập. Full notebook không tự chọn model tốt nhất bằng outer-test và không tự thay model deployment.

In [ ]:
import shutil
display(pd.read_csv(RUN_DIR / "pooled_by_seed.csv"))
archive = shutil.make_archive(str(RUN_DIR), "zip", RUN_DIR)
print("Download:", archive)
print("Next: return the ZIP for error analysis, paired ablation comparison, and proposal update with actual results.")